# 02. Uso de recursos: horizonte de predição e base de modelagem

**Base:** tabela `task_usage` do clusterdata-2011-2
**Entrada:** `saida/primeira_tentativa.parquet` (notebook 01)
**Saída:** `saida/base_modelagem.parquet`

Horizonte de predição **h = 10 minutos**: só entram as janelas de medição encerradas até `t_term − h`, sendo `t_term` o instante do evento terminal da primeira tentativa. Assim, o modelo usa apenas informação disponível pelo menos 10 minutos antes do desfecho. Os tempos de disco (`disk_io_time`, `max_disk_io`) são descartados porque deixam de ser registrados a partir do 15º dia.

## 1. Configuração do ambiente

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
import numpy as np
import pandas as pd

# Pasta de trabalho no Google Drive (dados intermediários e figuras)
BASE = '/content/drive/MyDrive/artigo_predicao'
for d in ['ev_parts', 'uso_parts', 'saida', 'figuras']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

KEY = ['job_id', 'task_index']   # identificador de uma tarefa no trace
FRAC = 10                        # amostra: jobs com job_id % 10 == 0 (~10% dos jobs)

## 2. Parâmetros e funções

In [ ]:
H = 600_000_000   # 10 minutos em microssegundos
URL_US = 'https://storage.googleapis.com/clusterdata-2011-2/task_usage/part-{:05d}-of-00500.csv.gz'
COLS_US = ['start_time','end_time','job_id','task_index','machine_id','cpu_rate','canonical_mem',
           'assigned_mem','unmapped_cache','total_cache','max_mem','disk_io_time','disk_space',
           'max_cpu','max_disk_io','cpi','mai','sample_portion','agg_type','sampled_cpu']
MEDIAS = ['cpu_rate','canonical_mem','assigned_mem','unmapped_cache','total_cache',
          'disk_space','cpi','mai','sampled_cpu']      # agregadas pela média
PICOS  = ['max_mem','max_cpu']                          # agregadas pelo máximo
USE_US = ['start_time','end_time','job_id','task_index'] + MEDIAS + PICOS
TIPOS_US = {c: 'float32' for c in MEDIAS + PICOS}
TIPOS_US.update(start_time='int64', end_time='int64', job_id='int64', task_index='int32')

prim = pd.read_parquet(f'{BASE}/saida/primeira_tentativa.parquet')
corte = prim[KEY + ['t_term']].copy()
corte['t_corte'] = corte.t_term - H
corte = corte.drop(columns='t_term')

def agrega(df):
    """Agregação parcial por tarefa: somas e contagens (para a média final) e máximos."""
    df = df.assign(dur=(df.end_time - df.start_time) / 1e6)   # duração da janela em segundos
    g = df.groupby(KEY)
    return pd.concat([g[MEDIAS].sum().add_suffix('_soma'),
                      g[MEDIAS].count().add_suffix('_n'),
                      g[PICOS].max(),
                      g.dur.sum().rename('tempo_obs'),
                      g.size().rename('n_janelas')], axis=1)

def processa_uso(i):
    out = f'{BASE}/uso_parts/u{i:03d}.parquet'
    if os.path.exists(out):
        return
    partes = []
    for ch in pd.read_csv(URL_US.format(i), header=None, names=COLS_US, usecols=USE_US,
                          dtype=TIPOS_US, chunksize=2_000_000):
        ch = ch[ch.job_id % FRAC == 0].merge(corte, on=KEY, how='inner')
        ch = ch[ch.end_time <= ch.t_corte]          # corte do horizonte
        if len(ch):
            partes.append(agrega(ch))
    res = pd.concat(partes).reset_index() if partes else pd.DataFrame(columns=KEY)
    res.to_parquet(out, index=False)

## 3. Teste com 2 arquivos

In [ ]:
for i in range(2):
    processa_uso(i)
t = pd.read_parquet(f'{BASE}/uso_parts/u000.parquet')
print(t.shape)
t.describe().T[['count', 'mean', 'max']]

## 4. Processamento dos 500 arquivos
Pode levar várias horas; se a sessão cair, rode a seção 1 e esta célula novamente.

In [ ]:
for i in range(500):
    for tentativa in range(3):
        try:
            processa_uso(i)
            break
        except Exception as e:
            print(i, e)
            time.sleep(10)

## 5. Consolidação da base de modelagem
Uma mesma tarefa aparece em vários arquivos; as somas, contagens e máximos parciais são combinados e só então a média é calculada.

In [ ]:
u = pd.concat([pd.read_parquet(f'{BASE}/uso_parts/u{i:03d}.parquet') for i in range(500)],
              ignore_index=True)
agg = {**{c + '_soma': 'sum' for c in MEDIAS}, **{c + '_n': 'sum' for c in MEDIAS},
       **{c: 'max' for c in PICOS}, 'tempo_obs': 'sum', 'n_janelas': 'sum'}
u = u.groupby(KEY).agg(agg)
for c in MEDIAS:
    u[c] = u[c + '_soma'] / u[c + '_n'].replace(0, np.nan)
u = u[MEDIAS + PICOS + ['tempo_obs', 'n_janelas']].reset_index()

base = prim.merge(u, on=KEY, how='inner')
base.to_parquet(f'{BASE}/saida/base_modelagem.parquet', index=False)

fora = prim.merge(u[KEY], on=KEY, how='left', indicator=True).query('_merge == "left_only"')
print('base:', len(base), '| falhas:', int(base.falha.sum()), f'({base.falha.mean():.4f})')
print('excluídas sem janela válida:', len(fora), f'| taxa de falha nelas: {fora.falha.mean():.4f}')
print('teste:', base[base.dia >= 24].falha.agg(['size', 'mean']).to_dict())
print(base.isna().mean().round(3).to_string())

## 6. Resultados obtidos (execução de 28/09/2026)

| Medida | Valor |
|---|---|
| Base de modelagem | 724.349 tarefas, 46.045 falhas (6,36%) |
| Teste (dias 24 a 29) | 134.744 tarefas, 6,73% de falha |
| Excluídas sem janela válida (menos de 10 min ou sem registro de uso) | 1.097.305, taxa de falha de 4,98% |
| Valores ausentes | CPI 17,4%; MAI 19,8% |

Consequência: o modelo cobre apenas tarefas com mais de 10 minutos de execução. CPI e MAI serão imputados pela mediana do treino, com indicador de ausência na classificação.